# Fine-Tune Qwen3.5-2B for Brain (Round 2 — Balanced V2 Dataset)

**Before running:** Go to `Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU` and click **Save**.

Fixes in V2:
1. **Context memory awareness** (`MC_03` fix: uses facts provided in context/memory, only refuses when info is truly absent)
2. **Strict `need_info` detection** (never hallucinates placeholder emails/files; outputs `status: need_info`)
3. **Exact word counts** (`IF_01` fix: exact 3 words, exact 1 word, exact 5 words)
4. **Crisp neutral refusals** (no hallucinated side-facts)
5. **3 Epochs** with slightly lower learning rate (`1.5e-4`) to prevent over-refusal

In [ ]:
# Cell 1: Verify GPU & Install Unsloth
!nvidia-smi
%%capture
!pip install -U unsloth

In [ ]:
# Cell 2: Upload train_data_v2.jsonl from your computer
from google.colab import files
uploaded = files.upload()
DATASET_PATH = list(uploaded.keys())[0]
print(f"Loaded dataset: {DATASET_PATH}")

In [ ]:
# Cell 3: Load Qwen3.5-2B in 4-bit & attach LoRA adapters
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.5-2B",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# Cell 4: Format dataset with Qwen chat template
from datasets import load_dataset

def format_prompts(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset = dataset.map(format_prompts, batched=True)
print(f"Training examples: {len(dataset)}")
print("Sample:\n", dataset[0]["text"][:400])

In [ ]:
# Cell 5: Fine-tune with SFTTrainer (3 epochs, balanced lr 1.5e-4)
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=1.5e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs_v2",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print("Training complete!", trainer_stats.metrics)

In [ ]:
# Cell 6: Verify key target cases
FastLanguageModel.for_inference(model)

test_cases = [
    ("You are Brain, a precise, honest local AI assistant.", "State the current status of the operating system in exactly three words."),
    ("You are Brain, a precise, honest local AI assistant. If relevant facts or memories are provided in the context, answer directly and accurately using them.", "Context Memory:\n- User City: Bangalore\n- User Role: Software Engineer\n\nQuestion: What city do I live in?"),
    ("You are the Brain Autonomous Task Planner. Decompose the user goal into an ordered, compact JSON plan.", "User Goal: Read logs/app.log, summarize errors, and email the summary to my manager."),
    ("You are Brain, a precise, honest local AI assistant.", "Who won the 2027 FIFA World Cup and what was the final score?"),
]

for sys_p, usr_p in test_cases:
    msgs = [{"role": "system", "content": sys_p}, {"role": "user", "content": usr_p}]
    inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.2)
    print("=== TEST:", usr_p.splitlines()[-1])
    print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
    print()

In [ ]:
# Cell 7: Export to GGUF (q4_k_m) and download
import glob
from google.colab import files

model.save_pretrained_gguf("qwen35_2b_brain_v2_gguf", tokenizer, quantization_method="q4_k_m")

# Search both possible directory patterns
candidates = (
    glob.glob("qwen35_2b_brain_v2_gguf*/*Q4_K_M.gguf") +
    glob.glob("qwen35_2b_brain_v2_gguf*/*.gguf") +
    glob.glob("*Q4_K_M.gguf")
)
print("Found GGUF candidates:", candidates)
if candidates:
    target = [c for c in candidates if "Q4_K_M" in c][0] if any("Q4_K_M" in c for c in candidates) else candidates[0]
    print(f"Downloading {target}...")
    files.download(target)
else:
    print("No GGUF file found directly. Please check the files sidebar.")